# Ensemble Chatbot (Mistral + Falcon) — Terminal-style

A high-quality ensemble chatbot running two open-source LLMs in parallel.

## Setup Steps

1. **Runtime** → Change runtime type → Hardware accelerator → **GPU** → Save (recommended).
2. Run each cell in order.
3. When prompted in the Hugging Face login cell, paste your HF token (input is hidden).
4. After models load, use the terminal loop at the end to chat. Type `exit` or `quit` to stop.

## Notes

- Models require a Hugging Face token with read access.
- If you run into out-of-memory errors, I can provide a quantized/bitsandbytes version using less VRAM.
- Responses use majority voting (if both models agree) or the longest response as tiebreaker.

In [ ]:
# Cell 1 - Install required libraries
!pip install -q transformers accelerate huggingface_hub

In [ ]:
# Cell 2 - Login to Hugging Face (paste token when prompted)
from getpass import getpass
from huggingface_hub import login

token = getpass("Paste your Hugging Face token (hidden): ")
login(token=token)
print("Logged in ✅")
HF_TOKEN = token

In [ ]:
# Cell 3 - Load two higher-quality models (may take a few minutes)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import time

device = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if device == 0 else "CPU (no GPU)")

# You can change these to other HF models if you prefer
MODEL_A = "mistralai/Mistral-7B-Instruct-v0.1"
MODEL_B = "tiiuae/falcon-7b-instruct"

def safe_load(model_name):
    print(f"Loading {model_name} ...")
    start = time.time()
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        use_auth_token=HF_TOKEN,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    gen = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device=device,
        framework="pt",
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    print(f"Loaded {model_name} in {time.time()-start:.1f}s")
    return gen

gen_a = safe_load(MODEL_A)
gen_b = safe_load(MODEL_B)
gens = [gen_a, gen_b]
print("All models ready. ✅")

In [ ]:
# Cell 4 - Helper functions to call models in parallel and combine outputs
from concurrent.futures import ThreadPoolExecutor
from collections import Counter

def call_gen(gen, prompt, max_new_tokens=180):
    """Call a single generator and extract text output."""
    out = gen(prompt, max_new_tokens=max_new_tokens, return_full_text=False)
    text = out[0].get("generated_text") or out[0].get("text") or str(out[0])
    return text.strip()

def ask_ensemble(prompt, timeout=None):
    """Query all models in parallel and return best answer via majority or longest."""
    with ThreadPoolExecutor(max_workers=len(gens)) as ex:
        futures = [ex.submit(call_gen, g, prompt) for g in gens]
        results = [f.result() for f in futures]
    
    cleaned = [r for r in results if r and r.strip()]
    
    if not cleaned:
        return {"answer": "", "raw": results, "strategy": "empty"}
    
    cnt = Counter(cleaned)
    top, freq = cnt.most_common(1)[0]
    
    if freq >= 2:
        return {"answer": top, "raw": results, "strategy": "majority"}
    
    longest = max(cleaned, key=len)
    return {"answer": longest, "raw": results, "strategy": "longest"}

In [ ]:
# Cell 5 - Terminal-style chat loop
print("\n" + "="*60)
print("Terminal chat ready. Type 'exit' or 'quit' to stop.")
print("="*60 + "\n")

while True:
    try:
        prompt = input("You: ").strip()
        
        if not prompt:
            continue
        
        if prompt.lower() in ("exit", "quit"):
            print("\nGoodbye. 👋")
            break
        
        print("\n[Querying both models...]\n")
        out = ask_ensemble(prompt)
        
        print(f"Bot (strategy: {out['strategy'].upper()}):")
        print("-" * 60)
        print(out["answer"])
        print("-" * 60)
        print("\n--- Raw model outputs ---")
        
        for i, r in enumerate(out["raw"], 1):
            print(f"\n[Model {i}]")
            print(r)
        
        print("\n")
    
    except KeyboardInterrupt:
        print("\n\nStopped by user. 🛑")
        break
    except Exception as e:
        print(f"Error: {e}")